# Создание выборки из уже скачанных parquet-чанков

Этот ноутбук **не скачивает данные заново** и **не вызывает модель**.  
Он берет уже сохраненные очищенные чанки из `CLEAN_DIR`, случайно выбирает `100` чанков с фиксированным `RANDOM_SEED`, затем из каждого выбранного чанка случайно берет до `2000` отзывов и сохраняет результат в отдельную директорию.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q pandas pyarrow tqdm


In [ ]:
import json
import random
import re
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# =========================
# Настройки
# =========================

BASE_DIR = "/content/drive/MyDrive/MLops_project/project"

# Директория с уже скачанными и очищенными parquet-чанками
# Она взята из ноутбука dataset_downloading.ipynb
CLEAN_DIR = Path(BASE_DIR) / "data/interim/wb_feedbacks_clean_full"

# Отдельная директория для новой случайной выборки
OUTPUT_DIR = Path(BASE_DIR) / "data/processed/random_100_chunks_2000_examples_seed_42"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Куда сохраняем результат
OUTPUT_PARQUET = OUTPUT_DIR / "sample_100_chunks_2000_each.parquet"
OUTPUT_CSV = OUTPUT_DIR / "sample_100_chunks_2000_each.csv"
SELECTED_CHUNKS_CSV = OUTPUT_DIR / "selected_chunks.csv"
METADATA_JSON = OUTPUT_DIR / "metadata.json"

TEXT_COL = "text"

RANDOM_SEED = 42
N_RANDOM_CHUNKS = 100
N_EXAMPLES_PER_CHUNK = 2000

print("CLEAN_DIR:", CLEAN_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


CLEAN_DIR: /content/drive/MyDrive/MLops_project/project/data/interim/wb_feedbacks_clean_full
OUTPUT_DIR: /content/drive/MyDrive/MLops_project/project/data/processed/random_100_chunks_2000_examples_seed_42


In [ ]:
# =========================
# Поиск уже скачанных parquet-файлов
# =========================

parquet_files = sorted(CLEAN_DIR.glob("clean_part_*.parquet"))

print(f"Найдено parquet-файлов: {len(parquet_files)}")

if len(parquet_files) == 0:
    raise FileNotFoundError(
        f"В директории {CLEAN_DIR} не найдено файлов clean_part_*.parquet. "
        "Проверь, что данные уже скачаны и путь CLEAN_DIR указан правильно."
    )


Найдено parquet-файлов: 1903


In [ ]:
# =========================
# Фиксированный выбор случайных чанков
# =========================

rng = random.Random(RANDOM_SEED)

n_chunks_to_take = min(N_RANDOM_CHUNKS, len(parquet_files))
selected_parquet_files = rng.sample(parquet_files, n_chunks_to_take)

selected_chunks_df = pd.DataFrame({
    "sample_order": list(range(len(selected_parquet_files))),
    "source_file": [p.name for p in selected_parquet_files],
    "source_path": [str(p) for p in selected_parquet_files],
})
selected_chunks_df.to_csv(SELECTED_CHUNKS_CSV, index=False, encoding="utf-8-sig")

print(f"Выбрано случайных чанков: {len(selected_parquet_files)}")
print(f"Список выбранных чанков сохранен в: {SELECTED_CHUNKS_CSV}")
selected_chunks_df.head()


Выбрано случайных чанков: 100
Список выбранных чанков сохранен в: /content/drive/MyDrive/MLops_project/project/data/processed/random_100_chunks_2000_examples_seed_42/selected_chunks.csv


,sample_order,source_file,source_path
0,0,clean_part_01309.parquet,/content/drive/MyDrive/MLops_project/project/d...
1,1,clean_part_00228.parquet,/content/drive/MyDrive/MLops_project/project/d...
2,2,clean_part_00051.parquet,/content/drive/MyDrive/MLops_project/project/d...
3,3,clean_part_01518.parquet,/content/drive/MyDrive/MLops_project/project/d...
4,4,clean_part_00563.parquet,/content/drive/MyDrive/MLops_project/project/d...


In [ ]:
# =========================
# Безопасное чтение parquet с таймаутом
# =========================

import signal
import pandas as pd

READ_TIMEOUT_SECONDS = 30

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException("Чтение parquet-файла заняло слишком много времени")

def read_parquet_with_timeout(path, columns=None, timeout_seconds=30):
    """
    Читает parquet-файл с ограничением по времени.
    Если чтение занимает больше timeout_seconds секунд — выбрасывает TimeoutException.
    """
    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout_seconds)

    try:
        df = pd.read_parquet(path, columns=columns)
        signal.alarm(0)
        return df
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

In [ ]:
# =========================
# Случайный выбор примеров из чанков с сохранением прогресса
# =========================

from pathlib import Path
import os
import re
import signal
import pandas as pd
from tqdm.auto import tqdm

# Сколько успешных чанков нужно получить
TARGET_SUCCESSFUL_CHUNKS = 100

# Сколько секунд максимум читаем один parquet
READ_TIMEOUT_SECONDS = 30

# Папка для промежуточных частей
PARTS_DIR = OUTPUT_DIR / "sampled_parts"
PARTS_DIR.mkdir(parents=True, exist_ok=True)

# Файл со статистикой по уже обработанным частям
ROWS_PER_CHUNK_PATH = OUTPUT_DIR / "rows_per_chunk.csv"


class TimeoutException(Exception):
    pass


def timeout_handler(signum, frame):
    raise TimeoutException("Чтение parquet-файла заняло слишком много времени")


def read_parquet_with_timeout(path, columns=None, timeout_seconds=30):
    """
    Читает parquet-файл с ограничением по времени.
    В Colab signal не всегда идеально работает с Google Drive,
    но в твоем случае он уже смог пропустить проблемный файл.
    """
    old_handler = signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(timeout_seconds)

    try:
        df = pd.read_parquet(path, columns=columns)
        signal.alarm(0)
        return df
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


# =========================
# Загружаем уже сохраненный прогресс
# =========================

# Проверяем, что Google Drive реально доступен
if not CLEAN_DIR.exists():
    raise RuntimeError(f"CLEAN_DIR недоступна: {CLEAN_DIR}. Перемонтируй Google Drive.")

if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not PARTS_DIR.exists():
    PARTS_DIR.mkdir(parents=True, exist_ok=True)

test_files = list(CLEAN_DIR.glob("clean_part_*.parquet"))
if len(test_files) == 0:
    raise RuntimeError(
        f"В CLEAN_DIR не найдено parquet-файлов: {CLEAN_DIR}. "
        "Скорее всего, Google Drive не подключен или путь неправильный."
    )

print("Drive доступен.")
print("Найдено исходных parquet-файлов:", len(test_files))
existing_part_files = sorted(PARTS_DIR.glob("sample_part_*.parquet"))

processed_source_files = set()
existing_parts = []

for part_path in existing_part_files:
    try:
        part_df = pd.read_parquet(part_path, columns=["source_file"])
        if len(part_df) > 0:
            source_file = part_df["source_file"].iloc[0]
            processed_source_files.add(source_file)
            existing_parts.append(part_path)
    except Exception as e:
        print(f"Не удалось прочитать уже сохраненную часть: {part_path.name}")
        print("Ошибка:", repr(e))

print("Уже успешно сохранено частей:", len(existing_parts))
print("Уже обработано исходных файлов:", len(processed_source_files))


# =========================
# Формируем случайный порядок всех parquet-файлов
# =========================

all_candidate_files = list(parquet_files)

rng = random.Random(RANDOM_SEED)
rng.shuffle(all_candidate_files)

print("Всего доступных parquet-файлов:", len(all_candidate_files))
# =========================
# Проблемные файлы, которые пропускаем сразу
# =========================

BAD_FILES = {
    "clean_part_00191.parquet",
    "clean_part_01720.parquet",
    "clean_part_00666.parquet",
}

print("Проблемные файлы будут пропущены:", BAD_FILES)

# =========================
# Основной цикл
# =========================

sampled_parts = []
rows_per_chunk = []

# Подгружаем старую статистику, если есть
if ROWS_PER_CHUNK_PATH.exists():
    rows_per_chunk_df_old = pd.read_csv(ROWS_PER_CHUNK_PATH)
    rows_per_chunk = rows_per_chunk_df_old.to_dict("records")
else:
    rows_per_chunk_df_old = None

successful_chunks = len(existing_parts)

for file_path in tqdm(all_candidate_files, desc="Sampling chunks"):
    # Если уже набрали 100 успешных чанков — останавливаемся
    if successful_chunks >= TARGET_SUCCESSFUL_CHUNKS:
        break

    # Если этот файл уже был успешно обработан раньше — пропускаем
    if file_path.name in processed_source_files:
        continue

    # Если файл заранее известен как проблемный — пропускаем
    if file_path.name in BAD_FILES:
        print(f"Пропускаю заранее проблемный файл: {file_path.name}")
        continue

    print(f"\nУспешных чанков: {successful_chunks}/{TARGET_SUCCESSFUL_CHUNKS}")
    print(f"Читаю файл: {file_path.name}")

    try:
        df = read_parquet_with_timeout(
            file_path,
            columns=[TEXT_COL],
            timeout_seconds=READ_TIMEOUT_SECONDS,
        )

    except TimeoutException:
        print(f"Пропускаю файл: {file_path.name}")
        print(f"Причина: чтение заняло больше {READ_TIMEOUT_SECONDS} секунд")
        continue

    except OSError as e:
        print(f"Ошибка чтения файла: {file_path.name}")
        print("Ошибка:", repr(e))

        # Если Drive отвалился или путь стал недоступен,
        # дальше продолжать бессмысленно: все следующие файлы тоже будут падать.
        if (
            "Transport endpoint is not connected" in str(e)
            or isinstance(e, FileNotFoundError)
            or "No such file or directory" in str(e)
        ):
            raise RuntimeError(
                "Google Drive недоступен или отвалился. "
                "Останови выполнение, перемонтируй Drive и запусти ячейки 3, 4, 7 заново. "
                "Папку sampled_parts не удаляй."
            )

        continue

    except Exception as e:
        print(f"Не удалось прочитать файл: {file_path.name}")
        print("Ошибка:", repr(e))
        continue

    print(f"Файл прочитан: {file_path.name}, shape={df.shape}")

    if TEXT_COL not in df.columns:
        print(f"Пропускаю файл без колонки {TEXT_COL}: {file_path.name}")
        continue

    # Убираем пустые отзывы
    df = df[df[TEXT_COL].notna()].copy()
    df[TEXT_COL] = df[TEXT_COL].astype(str)
    df = df[df[TEXT_COL].str.strip() != ""].copy()

    if len(df) == 0:
        print(f"Пропускаю пустой чанк: {file_path.name}")
        continue

    n_take = min(N_EXAMPLES_PER_CHUNK, len(df))

    df_sample = df.sample(
        n=n_take,
        replace=False,
        random_state=RANDOM_SEED + successful_chunks,
    ).copy()

    # Сохраняем исходный индекс строки внутри parquet-чанка
    df_sample.insert(0, "source_row_idx", df_sample.index)
    df_sample.insert(0, "source_file", file_path.name)
    df_sample.insert(0, "source_sample_order", successful_chunks)

    match = re.search(r"clean_part_(\d+)\.parquet$", file_path.name)
    source_chunk_id = int(match.group(1)) if match else None
    df_sample.insert(0, "source_chunk_id", source_chunk_id)

    df_sample = df_sample.reset_index(drop=True)

    part_path = PARTS_DIR / f"sample_part_{successful_chunks:05d}_{file_path.stem}.parquet"
    df_sample.to_parquet(part_path, index=False)

    processed_source_files.add(file_path.name)
    sampled_parts.append(df_sample)

    rows_per_chunk.append({
        "sample_order": successful_chunks,
        "source_file": file_path.name,
        "source_chunk_id": source_chunk_id,
        "rows_available_after_filter": len(df),
        "rows_sampled": len(df_sample),
        "part_path": str(part_path),
    })

    # Сохраняем статистику после каждого успешного чанка
    rows_per_chunk_df = pd.DataFrame(rows_per_chunk)
    rows_per_chunk_df.to_csv(ROWS_PER_CHUNK_PATH, index=False)

    successful_chunks += 1

    print(f"Сохранено {len(df_sample)} строк в {part_path.name}")


print("\nУспешно собрано чанков:", successful_chunks)

Drive доступен.
Найдено исходных parquet-файлов: 1903
Уже успешно сохранено частей: 57
Уже обработано исходных файлов: 57
Всего доступных parquet-файлов: 1903
Проблемные файлы будут пропущены: {'clean_part_01720.parquet', 'clean_part_00666.parquet', 'clean_part_00191.parquet'}


Sampling chunks:   0%|          | 0/1903 [00:00<?, ?it/s]

Пропускаю заранее проблемный файл: clean_part_01720.parquet
Пропускаю заранее проблемный файл: clean_part_00666.parquet

Успешных чанков: 57/100
Читаю файл: clean_part_00845.parquet
Файл прочитан: clean_part_00845.parquet, shape=(100000, 1)
Сохранено 2000 строк в sample_part_00057_clean_part_00845.parquet

Успешных чанков: 58/100
Читаю файл: clean_part_01509.parquet
Файл прочитан: clean_part_01509.parquet, shape=(100000, 1)
Сохранено 2000 строк в sample_part_00058_clean_part_01509.parquet

Успешных чанков: 59/100
Читаю файл: clean_part_00024.parquet
Файл прочитан: clean_part_00024.parquet, shape=(100000, 1)
Сохранено 2000 строк в sample_part_00059_clean_part_00024.parquet

Успешных чанков: 60/100
Читаю файл: clean_part_01847.parquet
Файл прочитан: clean_part_01847.parquet, shape=(100000, 1)
Сохранено 2000 строк в sample_part_00060_clean_part_01847.parquet

Успешных чанков: 61/100
Читаю файл: clean_part_00737.parquet
Файл прочитан: clean_part_00737.parquet, shape=(100000, 1)
Сохранено 2

In [ ]:
# =========================
# Сборка итогового файла из сохраненных частей
# =========================

part_files = sorted(PARTS_DIR.glob("sample_part_*.parquet"))

print("Найдено сохраненных частей:", len(part_files))

if len(part_files) == 0:
    raise RuntimeError("Не найдено ни одной сохраненной части.")

sample_df = pd.concat(
    [pd.read_parquet(p) for p in part_files],
    ignore_index=True,
)

rows_per_chunk_df = pd.read_csv(ROWS_PER_CHUNK_PATH)

print("Итоговый размер выборки:", sample_df.shape)
print("Количество успешных чанков:", len(part_files))
print("Ожидаемый максимум:", TARGET_SUCCESSFUL_CHUNKS * N_EXAMPLES_PER_CHUNK)

rows_per_chunk_df.head()

Найдено сохраненных частей: 100
Итоговый размер выборки: (200000, 5)
Количество успешных чанков: 100
Ожидаемый максимум: 200000


,sample_order,source_file,source_chunk_id,rows_available_after_filter,rows_sampled,part_path
0,19,clean_part_01109.parquet,1109,100000,2000,/content/drive/MyDrive/MLops_project/project/d...
1,20,clean_part_00979.parquet,979,100000,2000,/content/drive/MyDrive/MLops_project/project/d...
2,21,clean_part_00523.parquet,523,100000,2000,/content/drive/MyDrive/MLops_project/project/d...
3,22,clean_part_01257.parquet,1257,100000,2000,/content/drive/MyDrive/MLops_project/project/d...
4,23,clean_part_00731.parquet,731,100000,2000,/content/drive/MyDrive/MLops_project/project/d...


In [ ]:
# =========================
# Сохранение результата
# =========================

sample_df.to_parquet(OUTPUT_PARQUET, index=False)
sample_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

rows_per_chunk_df.to_csv(OUTPUT_DIR / "rows_per_chunk.csv", index=False, encoding="utf-8-sig")

metadata = {
    "base_dir": BASE_DIR,
    "clean_dir": str(CLEAN_DIR),
    "output_dir": str(OUTPUT_DIR),
    "random_seed": RANDOM_SEED,
    "n_random_chunks_requested": N_RANDOM_CHUNKS,
    "n_random_chunks_selected": len(selected_parquet_files),
    "n_examples_per_chunk_requested": N_EXAMPLES_PER_CHUNK,
    "total_rows_sampled": int(len(sample_df)),
    "text_col": TEXT_COL,
    "output_parquet": str(OUTPUT_PARQUET),
    "output_csv": str(OUTPUT_CSV),
    "selected_chunks_csv": str(SELECTED_CHUNKS_CSV),
}

with open(METADATA_JSON, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Готово")
print(f"Parquet: {OUTPUT_PARQUET}")
print(f"CSV: {OUTPUT_CSV}")
print(f"Список чанков: {SELECTED_CHUNKS_CSV}")
print(f"Метаданные: {METADATA_JSON}")


Готово
Parquet: /content/drive/MyDrive/MLops_project/project/data/processed/random_100_chunks_2000_examples_seed_42/sample_100_chunks_2000_each.parquet
CSV: /content/drive/MyDrive/MLops_project/project/data/processed/random_100_chunks_2000_examples_seed_42/sample_100_chunks_2000_each.csv
Список чанков: /content/drive/MyDrive/MLops_project/project/data/processed/random_100_chunks_2000_examples_seed_42/selected_chunks.csv
Метаданные: /content/drive/MyDrive/MLops_project/project/data/processed/random_100_chunks_2000_examples_seed_42/metadata.json


In [ ]:
# Быстрая проверка
print(sample_df.shape)
sample_df[["source_chunk_id", "source_file", "source_row_idx", TEXT_COL]].head()


(200000, 5)


,source_chunk_id,source_file,source_row_idx,text
0,1309,clean_part_01309.parquet,75721,"Качество хорошее, но мне влики, отказ"
1,1309,clean_part_01309.parquet,80184,"Волшебно! Все супер, берите, не пожалеете)"
2,1309,clean_part_01309.parquet,19864,❤️❤️❤️❤️❤️ просто прелесть
3,1309,clean_part_01309.parquet,76699,Все пучком
4,1309,clean_part_01309.parquet,92991,Очень вкусноооо))) Спасибо большое!
